# Notebook 22 — Phase Transition Finder

**Repository path:** `control-stack/22_phase_transition_finder.ipynb`

This notebook extends Notebook 21 from a safe certification sweep into a **phase-transition finder**: it increases adversarial attack strength until each controller crosses the 45° CGCS stability boundary.

Core question:

> At what attack strength does each policy first lose phase-lock?

Outputs are written into `figures/`, `results/`, and `docs/`, then bundled into a downloadable zip.


In [ ]:
# ============================================================
# Notebook 22 setup
# ============================================================

import json
import math
import zipfile
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_ID = "22"
TITLE = "Phase Transition Finder"
SLUG = "phase_transition_finder"
SEED = 9423

rng = np.random.default_rng(SEED)

ROOT = Path(".")
FIG_DIR = ROOT / "figures"
RESULTS_DIR = ROOT / "results"
DOCS_DIR = ROOT / "docs"
MATH_DIR = ROOT / "math"

for d in [FIG_DIR, RESULTS_DIR, DOCS_DIR, MATH_DIR]:
    d.mkdir(parents=True, exist_ok=True)

THRESHOLD = 1 / np.sqrt(2)  # 45° cosine threshold
POLICIES = [
    "none",
    "moving_average",
    "joint_kalman",
    "robust_gated_kalman",
    "constrained_mpc",
    "cgcs_invariance_control",
    "oracle",
]

T = 90
PULSE_T = np.linspace(0, 10, 400)
strength_grid = np.linspace(0.0, 5.0, 51)

print(f"Notebook {NOTEBOOK_ID}: {TITLE}")
print(f"threshold = {THRESHOLD:.6f}")
print(f"attack strengths = {strength_grid[0]:.1f} → {strength_grid[-1]:.1f} ({len(strength_grid)} points)")


## Helper functions

The simulation keeps the same high-level control-stack vocabulary used in earlier notebooks:

- `Ω` drift: frequency / phase-command drift
- `B` drift: amplitude / bias-command drift
- CGCS margin: `cosine similarity − 1/√2`
- failure: `margin < 0`

Notebook 22 intentionally makes the adversary strong enough to find real crossings.


In [ ]:
# ============================================================
# Synthetic hardware-control stack primitives
# ============================================================

def smooth(x, window=5):
    """Centered moving average with edge padding."""
    if window <= 1:
        return np.asarray(x).copy()
    pad = window // 2
    xp = np.pad(x, (pad, pad), mode="edge")
    kernel = np.ones(window) / window
    return np.convolve(xp, kernel, mode="valid")


def make_base_drifts(T=90):
    """Base slow drift traces for Ω and B."""
    t = np.arange(T)
    omega = 0.035*np.sin(2*np.pi*t/23) + 0.018*np.sin(2*np.pi*t/9)
    beta = 0.006 + 0.0009*t + 0.009*np.sin(2*np.pi*t/29)
    return omega, beta


def attack_profile(strength, T=90):
    """Deterministic adversarial profile scaled by strength.

    Designed to expose actual phase-transition boundaries beyond the safe range in Notebook 21.
    """
    t = np.arange(T)
    omega = np.zeros(T)
    beta = np.zeros(T)

    # Compact phase-shift window: main phase-break attack.
    win = (t >= 20) & (t <= 34)
    envelope = np.sin(np.linspace(0, np.pi, win.sum()))**1.35
    omega[win] += strength * 0.145 * envelope
    beta[win] += strength * 0.075 * envelope

    # Secondary correlated burst / model mismatch window.
    win2 = (t >= 50) & (t <= 66)
    envelope2 = np.sin(np.linspace(0, np.pi, win2.sum()))**1.2
    omega[win2] -= strength * 0.080 * envelope2
    beta[win2] += strength * 0.048 * envelope2

    # Late impulse probes recovery/re-entry.
    for loc, amp in [(73, 0.06), (78, -0.05)]:
        if 0 <= loc < T:
            omega[loc] += strength * amp
            beta[loc] -= strength * 0.035*np.sign(amp)

    return omega, beta


def measured_drifts(strength, T=90, seed=SEED):
    """Measured Ω/B traces under base drift + adversarial attack + deterministic measurement noise."""
    local_rng = np.random.default_rng(seed + int(round(strength*1000)))
    base_o, base_b = make_base_drifts(T)
    atk_o, atk_b = attack_profile(strength, T)
    true_o = base_o + atk_o
    true_b = base_b + atk_b

    # Noise increases mildly with attack strength.
    noise_scale = 0.008 + 0.0025*strength
    meas_o = true_o + local_rng.normal(0, noise_scale, T)
    meas_b = true_b + local_rng.normal(0, noise_scale*0.8, T)
    return true_o, true_b, meas_o, meas_b


def estimate_policy(policy, true_o, true_b, meas_o, meas_b, strength):
    """Return estimated Ω/B for a given controller policy."""
    T = len(true_o)
    if policy == "oracle":
        return true_o.copy(), true_b.copy()

    if policy == "none":
        # Baseline underreacts and lags during attacks.
        eo = smooth(meas_o, 13) * (0.58 - 0.025*min(strength, 4))
        eb = smooth(meas_b, 13) * (0.62 - 0.020*min(strength, 4))
        return eo, eb

    if policy == "moving_average":
        return smooth(meas_o, 9), smooth(meas_b, 9)

    if policy == "joint_kalman":
        # Recursive filter approximation: moderate adaptation.
        alpha = 0.30
        eo = np.zeros(T)
        eb = np.zeros(T)
        eo[0], eb[0] = meas_o[0], meas_b[0]
        for i in range(1, T):
            eo[i] = (1-alpha)*eo[i-1] + alpha*meas_o[i]
            eb[i] = (1-alpha)*eb[i-1] + alpha*meas_b[i]
        return eo, eb

    if policy == "robust_gated_kalman":
        alpha = 0.26
        gate = 0.075
        eo = np.zeros(T)
        eb = np.zeros(T)
        eo[0], eb[0] = meas_o[0], meas_b[0]
        for i in range(1, T):
            ro = meas_o[i] - eo[i-1]
            rb = meas_b[i] - eb[i-1]
            # Gate rejects large update residuals; good for spikes, fragile for sustained attacks.
            if abs(ro) < gate:
                eo[i] = eo[i-1] + alpha*ro
            else:
                eo[i] = eo[i-1] + 0.04*np.sign(ro)
            if abs(rb) < gate:
                eb[i] = eb[i-1] + alpha*rb
            else:
                eb[i] = eb[i-1] + 0.025*np.sign(rb)
        return eo, eb

    if policy == "constrained_mpc":
        # Previewed smoothing with bounded correction.
        eo = 0.78*smooth(meas_o, 5) + 0.22*smooth(meas_o, 11)
        eb = 0.78*smooth(meas_b, 5) + 0.22*smooth(meas_b, 11)
        max_step = 0.055
        for arr in [eo, eb]:
            for i in range(1, T):
                step = arr[i] - arr[i-1]
                if abs(step) > max_step:
                    arr[i] = arr[i-1] + np.sign(step)*max_step
        return eo, eb

    if policy == "cgcs_invariance_control":
        # Invariance controller: combines tracking with a stronger correction inside detected attack windows.
        eo = 0.64*smooth(meas_o, 3) + 0.36*smooth(meas_o, 9)
        eb = 0.64*smooth(meas_b, 3) + 0.36*smooth(meas_b, 9)

        # Residual-triggered correction toward measured signal, but bounded to avoid overfitting spikes.
        for i in range(1, T):
            local_energy = abs(meas_o[i] - smooth(meas_o, 9)[i]) + abs(meas_b[i] - smooth(meas_b, 9)[i])
            if local_energy > 0.035:
                gain = min(0.45, 0.22 + 0.035*strength)
                eo[i] = (1-gain)*eo[i] + gain*meas_o[i]
                eb[i] = (1-gain)*eb[i] + gain*meas_b[i]

        # Prevent command discontinuities.
        max_step = 0.075
        for arr in [eo, eb]:
            for i in range(1, T):
                step = arr[i] - arr[i-1]
                if abs(step) > max_step:
                    arr[i] = arr[i-1] + np.sign(step)*max_step
        return eo, eb

    raise ValueError(f"unknown policy: {policy}")


def response_waveform(omega, beta, block, t=PULSE_T):
    """Excited-state probability waveform for one calibration block."""
    phase = 0.55*omega[block] + 0.30*beta[block]
    amp = 1.0 - 0.45*abs(beta[block])
    y = 0.5 + 0.5*amp*np.sin(t + phase)**2
    return np.clip(y, 0, 1.05)


def policy_metrics(policy, strength):
    """Evaluate policy under one attack strength."""
    true_o, true_b, meas_o, meas_b = measured_drifts(strength)
    est_o, est_b = estimate_policy(policy, true_o, true_b, meas_o, meas_b, strength)

    # Error vectors and cosine stability.
    err = np.sqrt((est_o - true_o)**2 + (est_b - true_b)**2)
    # Convert command error to cosine-like alignment. Scale strength makes boundary possible.
    cos_sim = 1.0 - 1.85*err - 0.012*strength
    cos_sim = np.clip(cos_sim, 0.0, 1.0)
    margin = cos_sim - THRESHOLD

    # Response RMSE uses pulse waveform deviations.
    rmses = []
    for b in range(len(true_o)):
        target = response_waveform(true_o, true_b, b)
        pred = response_waveform(est_o, est_b, b)
        rmses.append(float(np.sqrt(np.mean((target - pred)**2))))
    rmses = np.array(rmses)

    failures = margin < 0
    recovery = 0
    if failures.any():
        idx = np.where(failures)[0]
        for start in idx:
            after = np.where(~failures[start:])[0]
            recovery = max(recovery, int(after[0]) if len(after) else int(len(failures)-start))

    return {
        "policy": policy,
        "strength": float(strength),
        "min_margin": float(np.min(margin)),
        "mean_margin": float(np.mean(margin)),
        "blocks_below_threshold": int(np.sum(failures)),
        "mean_rmse": float(np.mean(rmses)),
        "p95_rmse": float(np.percentile(rmses, 95)),
        "max_rmse": float(np.max(rmses)),
        "mean_recovery_time": float(recovery),
        "cos_sim": cos_sim,
        "margin": margin,
        "rmse_trace": rmses,
        "true_o": true_o,
        "true_b": true_b,
        "meas_o": meas_o,
        "meas_b": meas_b,
        "est_o": est_o,
        "est_b": est_b,
    }


## Run phase-transition sweep

For each controller policy, sweep adversarial strength from `0.0` to `5.0` and record the first strength where `min_margin < 0`.


In [ ]:
# ============================================================
# Run sweep
# ============================================================

records = []
trace_store = {}

for strength in strength_grid:
    for policy in POLICIES:
        m = policy_metrics(policy, strength)
        records.append({k: v for k, v in m.items() if not isinstance(v, np.ndarray)})
        trace_store[(policy, float(strength))] = m

sweep_df = pd.DataFrame(records)

summary_rows = []
for policy in POLICIES:
    d = sweep_df[sweep_df["policy"] == policy].copy()
    failed = d[d["min_margin"] < 0]
    if len(failed):
        critical_strength = float(failed.iloc[0]["strength"])
    else:
        critical_strength = float(strength_grid[-1])
    final = d.iloc[-1]
    summary_rows.append({
        "policy": policy,
        "critical_strength": critical_strength,
        "final_min_margin": float(final["min_margin"]),
        "final_mean_rmse": float(final["mean_rmse"]),
        "final_p95_rmse": float(final["p95_rmse"]),
        "final_blocks_below_threshold": int(final["blocks_below_threshold"]),
        "final_recovery_time": float(final["mean_recovery_time"]),
        "max_strength_tested": float(strength_grid[-1]),
    })

summary_df = pd.DataFrame(summary_rows).sort_values(
    ["critical_strength", "final_min_margin"], ascending=[False, False]
).reset_index(drop=True)

sweep_path = RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_transition_sweep.csv"
summary_path = RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_policy_summary.csv"
sweep_df.to_csv(sweep_path, index=False)
summary_df.to_csv(summary_path, index=False)

print(f"saved sweep: {sweep_path}")
print(f"saved summary: {summary_path}")
display(summary_df)


In [ ]:
# ============================================================
# Plot helpers
# ============================================================

FIGURES = []

def savefig(name):
    path = FIG_DIR / f"{NOTEBOOK_ID}_{SLUG}_{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    FIGURES.append(str(path))
    print(f"saved: {path}")
    plt.show()


def shade_attack_windows(ax):
    ax.axvspan(20, 34, alpha=0.16, label="main attack window")
    ax.axvspan(50, 66, alpha=0.10, label="secondary attack window")


In [ ]:
# ============================================================
# Figure 1: minimum margin vs attack strength
# ============================================================

plt.figure(figsize=(12, 7))
for policy in POLICIES:
    d = sweep_df[sweep_df["policy"] == policy]
    plt.plot(d["strength"], d["min_margin"], marker="o", label=policy)
plt.axhline(0, linestyle="--", label="threshold margin = 0")
plt.xlabel("attack strength multiplier")
plt.ylabel("minimum CGCS margin")
plt.title("Phase transition finder: minimum margin vs attack strength")
plt.legend(ncol=2)
savefig("min_margin_curves")


In [ ]:
# ============================================================
# Figure 2: failure phase diagram
# ============================================================

phase = []
for policy in POLICIES:
    d = sweep_df[sweep_df["policy"] == policy].sort_values("strength")
    phase.append((d["min_margin"].to_numpy() < 0).astype(float))
phase = np.array(phase)

plt.figure(figsize=(12, 6))
plt.imshow(phase, aspect="auto", interpolation="nearest", origin="lower",
           extent=[strength_grid.min(), strength_grid.max(), -0.5, len(POLICIES)-0.5])
plt.colorbar(label="failure: margin < 0")
plt.yticks(range(len(POLICIES)), POLICIES)
plt.xlabel("attack strength multiplier")
plt.ylabel("policy")
plt.title("Phase transition finder: failure phase diagram")
savefig("phase_diagram")


In [ ]:
# ============================================================
# Figure 3: critical strength ranking
# ============================================================

rank = summary_df.sort_values("critical_strength", ascending=False)
plt.figure(figsize=(11, 6))
plt.bar(rank["policy"], rank["critical_strength"])
plt.axhline(strength_grid[-1], linestyle="--", label="max swept strength")
plt.xticks(rotation=30, ha="right")
plt.ylabel("first failure strength; max if no failure")
plt.title("Phase transition finder: critical strength ranking")
plt.legend()
savefig("critical_strength")


In [ ]:
# ============================================================
# Select boundary and post-boundary strengths for waveform figures
# ============================================================

non_oracle = summary_df[summary_df["policy"] != "oracle"].copy()
strongest_policy = non_oracle.iloc[0]["policy"]
strongest_critical = float(non_oracle.iloc[0]["critical_strength"])

# boundary = just before failure, post = at/after failure if available
if strongest_critical <= strength_grid[0]:
    boundary_strength = float(strength_grid[0])
    post_strength = float(strength_grid[1])
elif strongest_critical >= strength_grid[-1]:
    boundary_strength = float(strength_grid[-1])
    post_strength = float(strength_grid[-1])
else:
    idx = int(np.where(np.isclose(strength_grid, strongest_critical))[0][0])
    boundary_strength = float(strength_grid[max(0, idx-1)])
    post_strength = float(strength_grid[idx])

post_metrics = trace_store[(strongest_policy, post_strength)]
worst_block = int(np.argmin(post_metrics["margin"]))

print("strongest non-oracle policy:", strongest_policy)
print("boundary strength:", boundary_strength)
print("post-boundary strength:", post_strength)
print("worst block:", worst_block)


In [ ]:
# ============================================================
# Figure 4: boundary waveform
# ============================================================

plt.figure(figsize=(13, 6))
base = trace_store[("oracle", boundary_strength)]
target = response_waveform(base["true_o"], base["true_b"], worst_block)
plt.plot(PULSE_T, target, linewidth=2.5, label="target")
for policy in POLICIES:
    m = trace_store[(policy, boundary_strength)]
    pred = response_waveform(m["est_o"], m["est_b"], worst_block)
    plt.plot(PULSE_T, pred, label=policy)
plt.xlabel("time / pulse duration")
plt.ylabel("excited-state probability")
plt.title(f"Phase transition finder: boundary waveform — block {worst_block}, strength {boundary_strength:.2f}")
plt.legend(ncol=2)
savefig("boundary_waveform")


In [ ]:
# ============================================================
# Figure 5: post-boundary waveform
# ============================================================

plt.figure(figsize=(13, 6))
base = trace_store[("oracle", post_strength)]
target = response_waveform(base["true_o"], base["true_b"], worst_block)
plt.plot(PULSE_T, target, linewidth=2.5, label="target")
for policy in POLICIES:
    m = trace_store[(policy, post_strength)]
    pred = response_waveform(m["est_o"], m["est_b"], worst_block)
    plt.plot(PULSE_T, pred, label=policy)
plt.xlabel("time / pulse duration")
plt.ylabel("excited-state probability")
plt.title(f"Phase transition finder: post-boundary waveform — block {worst_block}, strength {post_strength:.2f}")
plt.legend(ncol=2)
savefig("post_boundary_waveform")


In [ ]:
# ============================================================
# Figure 6: RMSE vs attack strength
# ============================================================

plt.figure(figsize=(12, 7))
for policy in POLICIES:
    d = sweep_df[sweep_df["policy"] == policy]
    plt.plot(d["strength"], d["mean_rmse"], marker="o", label=policy)
plt.xlabel("attack strength multiplier")
plt.ylabel("mean response RMSE")
plt.title("Phase transition finder: RMSE vs attack strength")
plt.legend(ncol=2)
savefig("rmse_curves")


In [ ]:
# ============================================================
# Figure 7: recovery time vs attack strength
# ============================================================

plt.figure(figsize=(12, 7))
for policy in POLICIES:
    d = sweep_df[sweep_df["policy"] == policy]
    plt.plot(d["strength"], d["mean_recovery_time"], marker="o", label=policy)
plt.xlabel("attack strength multiplier")
plt.ylabel("recovery blocks after failure")
plt.title("Phase transition finder: recovery time vs attack strength")
plt.legend(ncol=2)
savefig("recovery_curves")


In [ ]:
# ============================================================
# Figure 8: failure onset zoom
# ============================================================

zoom_strength = post_strength
plt.figure(figsize=(13, 6))
for policy in POLICIES:
    m = trace_store[(policy, zoom_strength)]
    plt.plot(np.arange(T), m["margin"], marker="o", label=policy)
plt.axhline(0, linestyle="--", label="threshold margin = 0")
shade_attack_windows(plt.gca())
plt.xlim(max(0, worst_block-15), min(T-1, worst_block+15))
plt.xlabel("calibration block")
plt.ylabel("CGCS margin above threshold")
plt.title(f"Phase transition finder: failure onset zoom at strength {zoom_strength:.2f}")
plt.legend(ncol=2)
savefig("failure_onset_zoom")


## Export report, manifest, README snippet, and zip bundle

This cell writes a markdown report and manifest, then zips all notebook outputs.


In [ ]:
# ============================================================
# Write markdown report, manifest, README snippet, and zip bundle
# ============================================================

figure_names = [
    "min_margin_curves",
    "phase_diagram",
    "critical_strength",
    "boundary_waveform",
    "post_boundary_waveform",
    "rmse_curves",
    "recovery_curves",
    "failure_onset_zoom",
]

best_policy = summary_df.iloc[0]["policy"]
weakest_policy = summary_df.iloc[-1]["policy"]

md_lines = []
md_lines.append(f"# Notebook {NOTEBOOK_ID} — {TITLE}")
md_lines.append("")
md_lines.append("## Summary")
md_lines.append("")
md_lines.append(
    "Notebook 22 extends the certification boundary sweep into an explicit phase-transition finder. "
    "It increases adversarial attack strength until each controller first crosses below the 45° CGCS margin."
)
md_lines.append("")
md_lines.append("## Policy transition summary")
md_lines.append("")
md_lines.append(summary_df.to_markdown(index=False))
md_lines.append("")
md_lines.append("## Interpretation")
md_lines.append("")
md_lines.append(
    f"The strongest non-oracle policy by first-failure strength is `{best_policy}`. "
    f"The weakest policy by the same ordering is `{weakest_policy}`. "
    "This notebook measures an empirical boundary: safe operation is not only summarized by accuracy, "
    "but by how far each controller can move along the adversarial strength axis before phase-lock fails."
)
md_lines.append("")
md_lines.append("## Figures")
md_lines.append("")
for name in figure_names:
    md_lines.append(f"![{name}](../figures/{NOTEBOOK_ID}_{SLUG}_{name}.png)")

md_path = DOCS_DIR / f"{NOTEBOOK_ID}_{SLUG}.md"
md_path.write_text("\n".join(md_lines))
print(f"saved markdown: {md_path}")

readme_snippet = f"""## Notebook {NOTEBOOK_ID} — {TITLE}

Phase-transition finder for the control stack. Sweeps adversarial attack strength from 0.0 to 5.0 and records each policy's first CGCS margin failure.

Primary outputs:

- `figures/{NOTEBOOK_ID}_{SLUG}_min_margin_curves.png`
- `figures/{NOTEBOOK_ID}_{SLUG}_phase_diagram.png`
- `figures/{NOTEBOOK_ID}_{SLUG}_critical_strength.png`
- `results/{NOTEBOOK_ID}_{SLUG}_transition_sweep.csv`
- `results/{NOTEBOOK_ID}_{SLUG}_policy_summary.csv`
- `docs/{NOTEBOOK_ID}_{SLUG}.md`
"""
readme_path = DOCS_DIR / f"{NOTEBOOK_ID}_{SLUG}_README_snippet.md"
readme_path.write_text(readme_snippet)
print(f"saved README snippet: {readme_path}")

manifest = {
    "notebook": f"{NOTEBOOK_ID}_{SLUG}",
    "title": TITLE,
    "seed": SEED,
    "threshold": THRESHOLD,
    "strength_grid": [float(x) for x in strength_grid],
    "policies": POLICIES,
    "best_policy": str(best_policy),
    "weakest_policy": str(weakest_policy),
    "figures": [str(FIG_DIR / f"{NOTEBOOK_ID}_{SLUG}_{name}.png") for name in figure_names],
    "results": [str(sweep_path), str(summary_path)],
    "docs": [str(md_path), str(readme_path)],
}
manifest_path = RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f"saved manifest: {manifest_path}")

zip_path = ROOT / f"{NOTEBOOK_ID}_{SLUG}_outputs.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in manifest["figures"] + manifest["results"] + manifest["docs"] + [str(manifest_path)]:
        p = Path(path)
        if p.exists():
            z.write(p, arcname=str(p))
print(f"saved zip: {zip_path}")


In [ ]:
# ============================================================
# Optional Colab download
# ============================================================

try:
    from google.colab import files
    files.download(f"{NOTEBOOK_ID}_{SLUG}_outputs.zip")
except Exception as exc:
    print("Colab download skipped:", exc)
    print(f"Zip is available at: {NOTEBOOK_ID}_{SLUG}_outputs.zip")
